In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('US Cancer Death Rates by State Machine.csv')
df = df.rename(columns={"Unnamed: 0": "State"})
print(df.head())
df = df.dropna()
X = df[['2014_Rate', '2015_Rate', '2016_Rate', '2017_Rate', '2018_Rate', '2019_Rate', '2020_Rate', '2021_Rate', '2022_Rate']]
y = df['2023_Rate']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training size:", X_train.shape)
print("Testing size:", X_test.shape)

  State  2014_Rate  2014_Total  2015_Rate  2015_Total  2016_Rate  2016_Total  \
0    AK      164.2       972.0      159.8       978.0      158.7       995.0   
1    AL      177.6     10286.0      175.6     10354.0      174.0     10419.0   
2    AR      183.1      6546.0      185.4      6727.0      178.8      6612.0   
3    AZ      142.7     11455.0      141.3     11776.0      136.8     11876.0   
4    CA      144.1     58412.0      142.8     59629.0      139.7     59515.0   

   2017_Rate  2017_Total  2018_Rate  ...  2019_Rate  2019_Total  2020_Rate  \
0      139.2       926.0      141.5  ...      146.9      1021.0      143.7   
1      170.0     10410.0      170.4  ...      160.8     10266.0      161.6   
2      173.6      6517.0      168.8  ...      165.7      6482.0      163.8   
3      135.8     12008.0      131.9  ...      131.1     12503.0      127.7   
4      136.7     59516.0      135.0  ...      131.6     59512.0      130.3   

   2020_Total  2021_Rate  2021_Total  2022_Rate  2

I picked the death rate of 2023 as my y value because the goal is to predict future values, but by using 2023 as the target rate, using 2014-2022 as the X value, we can find a pattern to make a model that can predict future values accurately.

In [9]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

dt_model = DecisionTreeRegressor(random_state=42)
dt_model.fit(X_train, y_train)
y_pred_dt = dt_model.predict(X_test)
mse_dt = mean_squared_error(y_test, y_pred_dt)
r2_dt = r2_score(y_test, y_pred_dt)

print("Decision Tree MSE:", mse_dt)
print("Decision Tree R2 Score:", r2_dt)

Decision Tree MSE: 20.056999999999995
Decision Tree R2 Score: 0.8795812932838218


An r^2 value of .8795 means that the model represents aproximately 88% of the value, meaning that it is a accurate model. The mean squared error is a little over 20 which is very accurate because if you take the square root of the mean squared error, it would be aproximately 4.5, meaning that the model is off by roughly 4.5 deaths per 100,000 people.

In [10]:
from sklearn.svm import SVR

svm_model = SVR(kernel='linear')
svm_model.fit(X_train, y_train)
y_pred_svm = svm_model.predict(X_test)
mse_svm = mean_squared_error(y_test, y_pred_svm)
r2_svm = r2_score(y_test, y_pred_svm)

print("SVM MSE:", mse_svm)
print("SVM R2 Score:",r2_svm)

SVM MSE: 12.33423084166444
SVM R2 Score: 0.9259474434715047


In [32]:
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix

y_binned = pd.qcut(df['2023_Rate'], q=3, labels=[0, 1, 2])
X_train, X_test, y_train, y_test = train_test_split(X, y_binned, test_size=0.2, random_state=42)

svm_classifier = SVC(kernel="rbf", C=1.0)
svm_classifier.fit(X_train, y_train)

y_predicted = svm_classifier.predict(X_test)
matrix = confusion_matrix(y_test, y_predicted)
results = pd.DataFrame({'Actual': y_test, 'DT_Predicted': y_pred_dt, 'SVM_Predicted': y_pred_svm})

print(results.head())
print("Confusion Matrix")
print(matrix)

   Actual  DT_Predicted  SVM_Predicted
14      1             1              1
40      2             1              2
31      0             0              0
46      1             1              1
18      2             2              2
Confusion Matrix
[[2 1 0]
 [0 3 0]
 [0 1 3]]


The results show that medium and low risks were predicted perfectly, but high risk was incorrectly predicted by the decision tree. the SVM model predicted all three very well though. For the confusion matrix, the diagonal values show that the model got 8 out of 10 correct. The remaining errors consist of two wrong answers where the model overestimated one low-risk state and underestimated one high-risk state as being medium-risk. The confusion matrix showed that the predictors are pretty accurate, but aren't 100% accurate.

By using cancer death rates from 2014–2022 to predict 2023 death rates, the model had a strong r^2 of 0.88 and an average error of only 4.5 deaths per 100,000 people. The decision tree had errors when it came to high-risk categories, but the SVM model performed reliably across all three risk levels. The confusion matrix showed this, by correctly identifying 8 out of 10 states with only two misses. In conclusion, these results prove that trends are highly effective at predicting outcomes.